# Сравнение с Prophet: 1, 3, 6 и 12 месяцев
Запуск в Google Colab: «Среда выполнения → Выполнить все». Загрузите пять исходных CSV СберИндекса (можно ZIP). Данные остаются в вашей сессии Colab. GPU не нужен. По умолчанию сравниваются 400 фиксированных муниципалитетов; для всех 2016 замените `--sample 400` на `--sample 0`.

Результаты этого ноутбука — наш воспроизводимый тест Prophet, не официальный результат организаторов. На 12 месяцах оценивается отдельно резервный метод при короткой истории. Варианты Prophet показываются все, без выбора по тестовым ошибкам. Для 400 рядов потребуется 13200 обучений Prophet; длительность зависит от среды. Ноутбук подготовлен локально, полный запуск с Prophet ещё не выполнен.


In [ ]:
!git clone --depth 1 --branch codex/forecasting-audit-improvements https://github.com/tenzerka2/SberIndex-Forecasting.git /content/SberIndex-Forecasting
%cd /content/SberIndex-Forecasting
%pip install -r requirements-prophet.txt


In [ ]:
from google.colab import files
from pathlib import Path
import io, zipfile
raw = Path('data/raw')
raw.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()
for name, content in uploaded.items():
    if name.lower().endswith('.zip'):
        with zipfile.ZipFile(io.BytesIO(content)) as archive:
            for member in archive.infolist():
                if member.filename.lower().endswith('.csv') and '__MACOSX' not in member.filename:
                    (raw / Path(member.filename).name).write_bytes(archive.read(member))
    elif name.lower().endswith('.csv'):
        (raw / Path(name).name).write_bytes(content)
print('CSV:', [p.name for p in raw.glob('*.csv')])


In [ ]:
import subprocess, sys
from datetime import datetime, timezone
run_dir = 'outputs/colab_prophet_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
subprocess.run([sys.executable, 'benchmarks/contest_prophet.py', '--workers', '2', '--sample', '400', '--output', run_dir], check=True)


In [ ]:
import pandas as pd
from IPython.display import display
result = Path(run_dir) / 'with_prophet'
display(pd.read_csv(result / 'metrics.csv'))
display(pd.read_csv(result / 'comparisons.csv'))


In [ ]:
import shutil
archive = shutil.make_archive('/content/prophet_benchmark_results', 'zip', result)
files.download(archive)
